# Laya → AmazingHand: Rock / Paper / Scissors
Chạy các cell từ trên xuống một lần. Sau đó mỗi lần chạy **cell cuối**: Laya suy luận local, in quyết định/xác suất, hiện kết quả và bàn tay thực hiện tư thế. Cell cuối cho bạn nhập nước để chơi với Laya.

- **Rock:** co cả index, middle, ring, thumb.
- **Paper:** duỗi cả bốn ngón.
- **Scissors:** duỗi index + middle, co ring + thumb.

Chọn kernel **AmazingHand (.venv)**, Python 3.12. Cell setup cài thư viện vào đúng kernel. Lần đầu cần Internet tải thư viện và weights (~808 MB); các lần sau dùng lại `models/laya-english/`. CPU chạy được; CUDA dùng được nếu PyTorch hiện tại hỗ trợ.

Model: [convaiinnovations/laya](https://huggingface.co/convaiinnovations/laya), [SDK Laya](https://github.com/NandhaKishorM/laya). Checkpoint tiếng Anh nên ngữ cảnh mặc định dùng tiếng Anh. Laya chưa được fine-tune cho RPS; xác suất không bảo đảm quyết định đúng. Cùng ngữ cảnh có thể chọn cùng một nước nhiều lần. Không thay quyết định model bằng nước ngẫu nhiên.

**Máy mới:** chọn Python 3.12, chạy các cell theo thứ tự. Cell setup tự cài dependency đúng kernel; cell load model tự tải Laya English khoảng 808 MB nếu chưa có. Lần đầu cần Internet; OFFLINE=True chỉ dùng khi checkpoint đã đủ. Cổng bàn tay tự chọn khi có đúng một USB serial; chỉnh PORT nếu nhiều thiết bị.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "PythonExample" / "AmazingHand_LayaRPS.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Mở notebook từ thư mục AmazingHand hoặc PythonExample.")
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
helper_path = str(PROJECT_ROOT / "PythonExample")
if helper_path not in sys.path:
    sys.path.insert(0, helper_path)
print("Project:", PROJECT_ROOT)
print("Kernel:", sys.executable)

from AmazingHand_Setup import ensure_dependencies, choose_serial_port


## 1. Cài môi trường
Chạy trước khi import torch/transformers/laya. Nếu đã import thư viện trước khi cài phiên bản khác, restart kernel rồi chạy lại từ đầu. Không cần cài toàn bộ stack mô phỏng.

In [ ]:
ensure_dependencies(PROJECT_ROOT / "requirements-laya.txt")
print("PASS — Dependencies đúng phiên bản trên kernel hiện tại.")

## 2. Cấu hình
`PORT` và các góc mặc định lấy từ notebook điều khiển hiện có. Đổi `PORT` nếu USB ở cổng khác. `DRY_RUN=False` để cell cuối thực hiện thật; `True` chỉ kiểm tra targets qua phần cứng và vẫn cần bàn tay kết nối.

Nhấn Interrupt khi cần dừng. Chạy `stop_hand()` trong một cell để nhả torque. `DECISION_CONTEXT` là đầu vào cho Laya, sau này thay bằng trạng thái đối thủ. Hiện model chỉ chọn một tư thế trình diễn.

In [ ]:
PORT = "auto"
DRY_RUN = False
PORT = choose_serial_port(PORT) if not DRY_RUN else PORT
SPEED_DEG_S = 80
OPEN_EXTENSIONS = (70, 60, 37, 35)  # index, middle, ring, thumb
CLOSE_FLEXIONS = (92, 90, 90, 90)
ALLOW_VOLTAGE_WARNING = True  # Giữ cấu hình notebook điều khiển hiện có.
END_TOLERANCE_DEG = 4
DEVICE = "auto"  # "auto", "cpu", "cuda"
OFFLINE = False  # True để chỉ dùng checkpoint đã có trên máy.
MODEL_DIR = PROJECT_ROOT / "models" / "laya-english"
DECISION_CONTEXT = (
    "Demonstrate one Rock, Paper, or Scissors gesture with the robot hand. "
    "There is no opponent yet. Choose one of the three available gestures."
)


## 3. Tải weights nếu chưa có và nạp Laya local
Chỉ tải checkpoint gốc tiếng Anh, không tải ảnh/tài liệu hoặc checkpoint khác. Pin revision để lần sau dùng cùng weights. File đánh dấu hoàn tất chỉ ghi sau khi tải xong. Nạp từ local ở chế độ Hub offline, không gọi dịch vụ inference bên ngoài.

In [ ]:
import numpy as np
import scipy
import scipy.sparse
from transformers.modeling_utils import no_init_weights

print("NumPy:", np.__version__, "SciPy:", scipy.__version__)
print("Import thành công. Nếu vừa đổi phiên bản dependencies, restart kernel trước khi nạp model.")


In [ ]:
import importlib
import time
import torch
import laya
import AmazingHand_LayaRPS as rps
from importlib.metadata import version

rps = importlib.reload(rps)
model_path = rps.ensure_model(MODEL_DIR, offline=OFFLINE)
device = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
if device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("PyTorch chưa có CUDA. Đặt DEVICE='cpu' hoặc cài PyTorch CUDA phù hợp.")
if device == "cpu":
    torch.set_num_threads(min(8, os.cpu_count() or 1))
model_key = (str(model_path), device, rps.MODEL_REVISION)
if globals().get("_loaded_model_key") != model_key:
    if "agent" in globals():
        del agent
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    started = time.perf_counter()
    previous_offline = os.environ.get("HF_HUB_OFFLINE")
    os.environ["HF_HUB_OFFLINE"] = "1"
    try:
        agent = laya.load(str(model_path), device=device, fast=False, compile=False)
    finally:
        if previous_offline is None:
            os.environ.pop("HF_HUB_OFFLINE", None)
        else:
            os.environ["HF_HUB_OFFLINE"] = previous_offline
    _loaded_model_key = model_key
    print(f"[MODEL] Nạp xong trong {time.perf_counter() - started:.2f} s")
print(f"[MODEL] {rps.MODEL_ID} @ {rps.MODEL_REVISION}")
print(f"[MODEL] Device: {agent.device}; laya={version('laya')}; torch={torch.__version__}")


## 4. Chuẩn bị bàn tay
Cell này chỉ tạo đối tượng điều khiển, chưa mở cổng hoặc chuyển động. Cell cuối kiểm tra servo, chạy một tư thế rồi giữ tư thế đó. Đóng kết nối từ notebook/app khác nếu đang giữ cùng cổng COM.

In [ ]:
import importlib
from serial.tools import list_ports

# Refresh all dependencies: an existing kernel must not retain the old parser.
for name in (
    "AmazingHand_HardwareDemo", "AmazingHand_Motion", "AmazingHand_GripCycles",
    "AmazingHand_VSign", "AmazingHand_Notebook", "AmazingHand_LayaRPS",
):
    module = importlib.import_module(name)
    importlib.reload(module)
import AmazingHand_LayaRPS as rps
from AmazingHand_Notebook import NotebookHand

print("Cổng serial:", [(p.device, p.description) for p in list_ports.comports()])
hand = NotebookHand(
    port=PORT, dry_run=DRY_RUN, speed_deg_s=SPEED_DEG_S,
    open_extensions=OPEN_EXTENSIONS, close_flexions=CLOSE_FLEXIONS,
    end_tolerance=END_TOLERANCE_DEG, motion_profile="smooth",
    allow_voltage_warning=ALLOW_VOLTAGE_WARNING, side="right", spread=0,
)
if "round_history" not in globals():
    round_history = []

def stop_hand():
    hand.release()

def tune_open(finger, delta=1):
    """Chỉnh riêng điểm duỗi 1–2 độ ở tốc độ chậm, rồi cập nhật cả manual/infer."""
    global OPEN_EXTENSIONS
    feedback = hand.adjust_endpoint(finger, "open", delta=delta)
    OPEN_EXTENSIONS = hand.open_extensions
    print("OPEN_EXTENSIONS =", OPEN_EXTENSIONS)
    return feedback

print(f"[HAND] {PORT}; tốc độ tối đa {hand.speed_deg_s:g}°/s; dry_run={DRY_RUN}")
print("[HAND] Điểm duỗi index/middle/ring/thumb:", hand.open_extensions)
print("[HAND] Chưa chuyển động. Paper và Scissors dùng chung điểm duỗi.")


### Test Rock / Paper / Scissors
Chạy riêng từng cell bên dưới để bàn tay thực hiện tư thế tương ứng, không cần Laya suy luận. Cần chạy cell chuẩn bị bàn tay trước. Bàn tay giữ tư thế sau khi chạy; gọi `stop_hand()` để nhả torque.

**Điểm duỗi mới:** `(70, 60, 37, 35)` cho index/middle/ring/thumb. Chỉ tăng index và middle theo phản hồi của bạn. Tốc độ yêu cầu **80°/s**, bộ điều khiển vẫn giảm nhịp nếu servo chưa theo kịp.

Nếu middle vẫn chưa thẳng, chạy riêng `tune_open("middle", delta=1)` rồi quan sát trước khi tăng tiếp. Có thể giảm bằng `delta=-1`. Mỗi bước tối đa 2°, chạy chậm và xác nhận phản hồi trước khi cập nhật điểm duỗi cho cả cell chạy tay và cell Laya. Index đã ở giới hạn phần mềm 70°; nếu vẫn cong, cần kiểm tra lại căn tâm/tay đòn cơ khí, không tăng vượt giới hạn. Servo đạt góc không bảo đảm ngón đã thẳng cơ học.


In [ ]:
# ROCK: Curl all four fingers. No Laya inference.
print("[HAND] ROCK: Curl all four fingers ...", flush=True)
hand.close_all()
print("[HAND] DRY RUN: targets checked; no movement." if hand.dry_run
      else "[HAND] ROCK: pose reached; holding torque.", flush=True)


In [ ]:
# PAPER: Extend all four fingers. No Laya inference.
print("[HAND] PAPER: Extend all four fingers ...", flush=True)
hand.open_all()
print("[HAND] DRY RUN: targets checked; no movement." if hand.dry_run
      else "[HAND] PAPER: pose reached; holding torque.", flush=True)


In [ ]:
# SCISSORS: Extend index and middle; curl ring and thumb. No Laya inference.
print("[HAND] SCISSORS: Extend index and middle; curl ring and thumb ...", flush=True)
hand.scissors()
print("[HAND] DRY RUN: targets checked; no movement." if hand.dry_run
      else "[HAND] SCISSORS: pose reached; holding torque.", flush=True)


## 5. CHƠI VỚI LAYA — nhập nước của bạn rồi chạy bàn tay
Chạy cell cuối: ô nhập xuất hiện, gõ **rock/paper/scissors**, **búa/bao/kéo** hoặc **1/2/3**, rồi Enter. Laya nhận nước bạn nhập và chọn nước đối đầu; đây là chế độ cho model biết nước của bạn trước. Chưa phải chế độ ra đồng thời.

Cell hiện nước của bạn, nước Laya, xác suất, thắng/thua/hòa, rồi thực hiện tư thế của Laya trên bàn tay. Laya tự quyết định; code chỉ chấm kết quả, không sửa nước model nếu model chọn sai. Bảng điểm chỉ tính lượt thực hiện thành công trên bàn tay thật; DRY RUN và lượt lỗi không cộng điểm.

Chạy lại cell để chơi lượt tiếp. Kết quả ở `last_round`, lịch sử ở `round_history`. Sơ đồ là minh họa, không phải ảnh camera. Chạy `stop_hand()` để nhả torque.


In [ ]:
from IPython.display import HTML, display
import importlib
import AmazingHand_LayaRPS as rps

rps = importlib.reload(rps)
last_round = None
PLAYER_MOVE = rps.normalize_move(input(
    "Nước của bạn [1=rock/búa, 2=paper/bao, 3=scissors/kéo]: "
))

_result_display = None
def show_result(result):
    global _result_display
    card = HTML(rps.result_html(result))
    if _result_display is None:
        _result_display = display(card, display_id=True)
    else:
        _result_display.update(card)

last_round = rps.run_round(
    agent=agent,
    hand=hand,
    context="Play Rock Paper Scissors against the human. Choose a winning robot move.",
    history=round_history,
    display_result=show_result,
    player_move=PLAYER_MOVE,
)
scores = rps.scoreboard(round_history)
print(f"[RESULT] Bạn: {PLAYER_MOVE.upper()} | Laya: {last_round['move'].upper()} | "
      f"{rps.OUTCOMES[last_round['outcome']]}")
print(f"[SCORE] Bạn {scores['player_win']} — Laya {scores['laya_win']} — Hòa {scores['draw']}")
